In [10]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [11]:
import warnings
warnings.filterwarnings("ignore")

In [12]:
def trans_donnees(X):
    X1 = X.copy()
    
    # Suppression des variables avec trop de valeurs manquantes (>50%)
    X1.drop(columns=[
        'Numero_Conteneur_Entree', 'Salle_Balance_Entree', 'Region_Entree', 'IP_Entree',
        'Numero_Conteneur_Sortie', 'ID_Balance_Sortie', 'Region_Sortie', 'IP_Sortie',
        'Routine_Manutention', 'Numero_Sequence', 'Remarque', 'Annule', 'Annule_Par', 'Date_Annulation'
    ], inplace=True, errors='ignore')
    
    # Suppression des identifiants inutiles (aucune valeur prédictive)
    X1.drop(columns=[
        'Numero_Camion', 'Plaque_Camion', 'ID_Cargaison',
        'Connaissement', 'Numero_Marque', 'Numero_Pesee', 'Voyage', 'Statut'
    ], inplace=True, errors='ignore')
    
    # Suppression des doublons
    X1 = X1.drop_duplicates()
    
    # Suppression valeurs aberrantes (variables quantitatives)
    X1.loc[X1['Poids_Cargaison_kg'] > 100000, 'Poids_Cargaison_kg'] = np.nan
    X1.loc[X1['Poids_Tare_kg'] > 100000, 'Poids_Tare_kg'] = np.nan
    X1.loc[X1['Poids_Camion_Entree_kg'] > 100000, 'Poids_Camion_Entree_kg'] = np.nan
    
    # Seuil surestarie — 10 000 min (~7 jours)
    X1.loc[X1['Surestarie_min'] > 10000, 'Surestarie_min'] = np.nan
    X1.loc[X1['Surestarie_min'] < 0, 'Surestarie_min'] = np.nan
    
    # Suppression variables redondantes et inutiles
    X1.drop(columns=[
        'Poids_Brut_Camion_kg',
        'Poids_Camion_Sortie_kg'  
    ], inplace=True, errors='ignore')
    
    # Suppression des lignes sans variable cible
    X1.dropna(subset=['Surestarie_min'], inplace=True)
    
    # Transformation logarithmique
    # ✅ Poids_Camion_Sortie_kg remplacé par Poids_Camion_Entree_kg
    log_vars = ['Surestarie_min', 'Poids_Tare_kg',
                'Poids_Cargaison_kg', 'Poids_Camion_Entree_kg']
    for var in log_vars:
        X1[f'log_{var}'] = np.log1p(X1[var])
    
    # Suppression du log de la cible (pas une feature)
    X1.drop(columns=['log_Surestarie_min'], inplace=True)
    
    # Feature engineering temporel
    X1['Heure_Entree_Gate'] = pd.to_datetime(X1['Heure_Entree_Gate'], format='%d/%m/%Y %H:%M')
    X1['Heure_Sortie_Gate'] = pd.to_datetime(X1['Heure_Sortie_Gate'], format='%d/%m/%Y %H:%M')
    
    X1['heure_entree'] = X1['Heure_Entree_Gate'].dt.hour
    X1['jour_semaine'] = X1['Heure_Entree_Gate'].dt.dayofweek
    X1['mois']         = X1['Heure_Entree_Gate'].dt.month
    
    X1.drop(columns=['Heure_Entree_Gate', 'Heure_Sortie_Gate'], inplace=True)
    
    # Typage des variables qualitatives
    var_quali = ['Nom_Cargaison', 'Nom_Navire', 'Operateur_Entree', 
                 'Operateur_Sortie', 'Type_Travail']
    for var in var_quali:
        X1[var] = X1[var].astype('object')
    
    return X1

In [13]:
train = pd.read_csv('C:/Users/LENOVO/Documents/Stage_DMP/port-ai-project/data/train.csv', index_col=0)
print(train.shape)
print(train.head())
print(train.columns)

(25498, 34)
         Numero_Camion Plaque_Camion ID_Cargaison   Connaissement  \
Statut                                                              
Gate-out    WDJB911D61     DJB911D61  DC240305005               2   
Gate-out  VCH042677500   CH042677500  DC240219002       TABUK0001   
Gate-out  VCH043848864   CH043848864  DC230909091   NSSCB23002351   
Gate-out     WETA18383      ETA18383  DC240224088  HK23170XDBT016   
Gate-out     WETA22810      ETA22810  DC240505007  NJ2401LYGDJ202   

                                   Numero_Marque  Nom_Cargaison   Nom_Navire  \
Statut                                                                         
Gate-out                      MILLINGWHEATINBULK          Wheat  BOS BOUTROS   
Gate-out                       JTMAA7BJ4P4063170        Vehicle  BAHRI TABUK   
Gate-out                       RKLKABAG3P0509459        Vehicle      JIGJIGA   
Gate-out  CCCC-FHECETHIOPIAEAHPROJECTVIADJIBOUTI         *Rebar     HELENA K   
Gate-out          BELTCO

In [14]:
train = trans_donnees(train)
X_train = train.drop(['Surestarie_min'], axis=1)
y_train = train['Surestarie_min']

In [15]:
val = pd.read_csv('C:/Users/LENOVO/Documents/Stage_DMP/port-ai-project/data/validation.csv', index_col=0)
print(val.head())

         Numero_Camion Plaque_Camion ID_Cargaison   Connaissement  \
Statut                                                              
Gate-out    WDJB288D62     DJB288D62  DC240214010   SSMCB24000009   
Gate-out      WET88021       ET88021  DC240114020   SSMCB24000002   
Gate-out  VCH042736646   CH042736646  DC240224043  HK23170LDBT044   
Gate-out     WETA10211      ETA10211  DC240123018   SSMCB24000005   
Gate-out     WET526D67      ET526D67  DC231121003        MZDDJI01   

                             Numero_Marque Nom_Cargaison    Nom_Navire Voyage  \
Statut                                                                          
Gate-out                    NPSBFERTILIZER    Fertilizer        ZURICH  12024   
Gate-out                    NPSBFERTILIZER    Fertilizer        SABAEK  04L23   
Gate-out                 LZZ5ELSC8PN256285       Vehicle      HELENA K  23170   
Gate-out                     NPSFERTILIZER    Fertilizer       CORINNA     49   
Gate-out  NONALLOYSTEELBILLETS

In [16]:
val = trans_donnees(val)
X_val = val.drop(['Surestarie_min'], axis=1)
y_val = val['Surestarie_min']

In [17]:
var_quali = ['Nom_Cargaison', 'Nom_Navire', 'Operateur_Entree', 
             'Operateur_Sortie', 'Type_Travail']

var_quanti = ['Poids_Tare_kg', 'Poids_Cargaison_kg', 
              'Poids_Camion_Entree_kg',          
              'heure_entree', 'jour_semaine', 'mois',
              'log_Poids_Tare_kg', 'log_Poids_Cargaison_kg', 
              'log_Poids_Camion_Entree_kg']       

# sans Surestarie_min puisque c'est la variable à prédire
var_moy = ['Poids_Tare_kg', 'Poids_Camion_Entree_kg', 
           'heure_entree', 'jour_semaine', 'mois']

var_med = ['Poids_Cargaison_kg', 'Poids_Camion_Entree_kg',  
           'log_Poids_Tare_kg', 'log_Poids_Cargaison_kg', 
           'log_Poids_Camion_Entree_kg']                     

### Imputation Simple

In [18]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
med_pipeline = Pipeline([
    ('imputer_med', SimpleImputer(strategy="median")),
    ('scaler_med', StandardScaler())
])
moy_pipeline = Pipeline([
    ('imputer_med', SimpleImputer(strategy="mean")),
    ('scaler_med', StandardScaler())
])
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('med', med_pipeline, var_med),
    ('moy', moy_pipeline, var_moy),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_simple_KNN = Pipeline([
    ('comp', preprocessor_simple),
    ('KNN', KNeighborsRegressor())
])

In [19]:
#Entrainement
pipe_simple_KNN.fit(X_train, y_train)
#prediction sur train
y_pred_train = pipe_simple_KNN.predict(X_train)
#prediction sur test
y_pred_val = pipe_simple_KNN.predict(X_val)

In [20]:
def qualite_reg(y, y_pred):
    mse = mean_squared_error(y, y_pred)
    print("MSE:", mse)
    print("RMSE", np.sqrt(mse))
    print("R²:", r2_score(y, y_pred))
    print("MAE", mean_absolute_error(y, y_pred))

In [21]:
#résultat obtenus en apprentissage
print("apprentissage")
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 241642.4095049661
RMSE 491.57136766187483
R²: 0.7011023298328218
MAE 203.38816805927792
validation
MSE: 334554.2536352201
RMSE 578.4066507529284
R²: 0.5369588832219194
MAE 256.36729559748426


In [22]:
#Grille d'hyperparamètres
param_grid = {
    "KNN__n_neighbors": [5, 7, 10, 15, 20, 25, 30], #nombre de voisins
    "KNN__p": [1, 2] } #type de distance 1:Manhattan

In [23]:
#GridSearchCV
grid = GridSearchCV( 
    estimator=pipe_simple_KNN, 
    param_grid=param_grid, 
    cv=5, 
    scoring="neg_mean_squared_error", 
    n_jobs=-1 ) 

In [24]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('comp',
                                        ColumnTransformer(transformers=[('med',
                                                                         Pipeline(steps=[('imputer_med',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler_med',
                                                                                          StandardScaler())]),
                                                                         ['Poids_Cargaison_kg',
                                                                          'Poids_Camion_Entree_kg',
                                                                          'log_Poids_Tare_kg',
                                                                          'log_Poids_Cargaison_kg',
                                                                          'log_Poids_Camion_Entree_kg']),
                                                                        ('moy',
                                                                         Pipeline(steps=[('imputer_med'...
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore'))]),
                                                                         ['Nom_Cargaison',
                                                                          'Nom_Navire',
                                                                          'Operateur_Entree',
                                                                          'Operateur_Sortie',
                                                                          'Type_Travail'])])),
                                       ('KNN', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'KNN__n_neighbors': [5, 7, 10, 15, 20, 25, 30],
                         'KNN__p': [1, 2]},
             scoring='neg_mean_squared_error')

In [25]:
print("Meilleurs paramètres :", grid.best_params_) 
print("Meilleure performance :", -grid.best_score_)

Meilleurs paramètres : {'KNN__n_neighbors': 5, 'KNN__p': 1}
Meilleure performance : 378504.83044650435


In [26]:
best_m = grid.best_estimator_
#résultat obtenus en apprentissage
print("apprentissage")
y_pred_train = best_m.predict(X_train)
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
y_pred_val = best_m.predict(X_val)
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 226207.0069320511
RMSE 475.61224430417167
R²: 0.720195029150745
MAE 198.4716774396973
validation
MSE: 316385.796
RMSE 562.4818183728253
R²: 0.5621050077208187
MAE 249.30528301886793


### Imputation KNN

In [27]:
#avec une imputation KNN
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
num_pipeline = Pipeline([
    ('KNN_imput_num', KNNImputer(n_neighbors=10)),
    ('scaler', StandardScaler())
])
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('num', num_pipeline, var_quanti),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_KNN_KNN = Pipeline([
    ('comp', preprocessor_simple),
    ('KNN', KNeighborsRegressor())
])

In [28]:
#Entrainement
pipe_KNN_KNN.fit(X_train, y_train)
#prediction sur train
y_pred_train = pipe_KNN_KNN.predict(X_train)
#prediction sur test
y_pred_val = pipe_KNN_KNN.predict(X_val)

In [29]:
#résultat obtenus en apprentissage
print("apprentissage")
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 233208.17002207154
RMSE 482.9163178254298
R²: 0.7115349957553051
MAE 201.2492747911083
validation
MSE: 327572.3716981132
RMSE 572.3393850663374
R²: 0.5466221840893944
MAE 257.74792452830184


In [30]:
#Grille d'hyperparamètres
param_grid = {
    "KNN__n_neighbors": [5, 7, 10, 15, 20, 25, 30],
    "KNN__p": [1, 2] }

In [31]:
#GridSearchCV
grid = GridSearchCV( 
    estimator=pipe_KNN_KNN, 
    param_grid=param_grid, 
    cv=5, 
    scoring="neg_mean_squared_error", 
    n_jobs=-1 ) 

In [32]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('comp',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('KNN_imput_num',
                                                                                          KNNImputer(n_neighbors=10)),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Poids_Tare_kg',
                                                                          'Poids_Cargaison_kg',
                                                                          'Poids_Camion_Entree_kg',
                                                                          'heure_entree',
                                                                          'jour_semaine',
                                                                          'mois',
                                                                          'log_Poids_Tare_kg',
                                                                          'log_Poids_Cargaison_kg',
                                                                          'log_Poids_Camion_Entre...
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore'))]),
                                                                         ['Nom_Cargaison',
                                                                          'Nom_Navire',
                                                                          'Operateur_Entree',
                                                                          'Operateur_Sortie',
                                                                          'Type_Travail'])])),
                                       ('KNN', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'KNN__n_neighbors': [5, 7, 10, 15, 20, 25, 30],
                         'KNN__p': [1, 2]},
             scoring='neg_mean_squared_error')

In [33]:
print("Meilleurs paramètres :", grid.best_params_) 
print("Meilleure performance :", -grid.best_score_)

Meilleurs paramètres : {'KNN__n_neighbors': 5, 'KNN__p': 1}
Meilleure performance : 366764.66963888257


In [34]:
best_m = grid.best_estimator_
#résultat obtenus en apprentissage
print("apprentissage")
y_pred_train = best_m.predict(X_train)
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
y_pred_val = best_m.predict(X_val)
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 221566.43148194862
RMSE 470.70843574547143
R²: 0.725935152306743
MAE 195.92923695412267
validation
MSE: 298524.92111949687
RMSE 546.3743415640022
R²: 0.5868254211109867
MAE 245.92685534591195


### Imputation Iteratives

In [35]:
#Avec une imputation iterative : bayesianRidge
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
num_pipeline = Pipeline([
    ('num_BR', IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=2026)),
    ('scaler', StandardScaler())
])
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('num', num_pipeline, var_quanti),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_IIBR_KNN = Pipeline([
    ('comp', preprocessor_simple),
    ('KNN', KNeighborsRegressor())
])

In [36]:
#Entrainement
pipe_IIBR_KNN.fit(X_train, y_train)
#prediction sur train
y_pred_train = pipe_IIBR_KNN.predict(X_train)
#prediction sur test
y_pred_val = pipe_IIBR_KNN.predict(X_val)

In [37]:
#résultat obtenus en apprentissage
print("apprentissage")
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 238036.4168075043
RMSE 487.88975886721
R²: 0.7055627340231261
MAE 200.80197855904146
validation
MSE: 333440.62018867926
RMSE 577.4431748567813
R²: 0.5385002119276963
MAE 254.467106918239


In [38]:
#Grille d'hyperparamètres
param_grid = {
    "KNN__n_neighbors": [5, 7, 10, 15, 20, 25, 30],
    "KNN__p": [1, 2] }

In [39]:
#GridSearchCV
grid = GridSearchCV( 
    estimator=pipe_IIBR_KNN, 
    param_grid=param_grid, 
    cv=5, 
    scoring="neg_mean_squared_error", 
    n_jobs=-1 ) 

In [40]:
grid.fit(X_train, y_train)
print("Meilleurs paramètres :", grid.best_params_) 
print("Meilleure performance :", -grid.best_score_)

Meilleurs paramètres : {'KNN__n_neighbors': 5, 'KNN__p': 1}
Meilleure performance : 375667.50273732375


In [41]:
best_m = grid.best_estimator_
#résultat obtenus en apprentissage
print("apprentissage")
y_pred_train = best_m.predict(X_train)
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
y_pred_val = best_m.predict(X_val)
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 224396.7382594987
RMSE 473.7053285107723
R²: 0.7224342266894179
MAE 197.06952546113826
validation
MSE: 314965.47753459116
RMSE 561.2178521167971
R²: 0.5640708050205311
MAE 250.08075471698115


#### Conclusion

In [ ]:
# Meilleur modèle : KNN avec imputation KNN + hyperparamètres optimisés.
# Résultats clés : RMSE validation = 546.4 min, R² = 0.587.
# Le tuning améliore significativement les performances
# (R² passe de 0.547 à 0.587, RMSE de 572.3 à 546.4).
# Surapprentissage : léger — R² train (0.726) > R² val (0.587),
# écart de 0.139 → le modèle généralise correctement mais mémorise un peu.
# Limite principale : un R² de 0.587 signifie que le modèle explique
# seulement 58.7% de la variance de la surestarie — correct mais insuffisant
# pour un usage en production.
# Recommandation : passer aux modèles ensemblistes, notamment la Forêt Aléatoire,
# qui gèrent mieux les non-linéarités et les interactions entre variables que le KNN.